## Author: Gabriel Vance
## Class: DSC550 - Data Mining
## Professor: Dr. Brett Werner
## Date: June 9, 2025

# Purpose: This notebook performs the tasks outlined in 1.2 Assignment: R/Python Refresher.


Requirements

You must use all 3 datasets provided to provide a visual story with the data with whatever medium you are choosing to present your story and remember it must include a call to action.
This exercise can use PowerBI/Tableau/R or Python. Make sure to carefully check each exercise, as some exercises will require a specific tool be used.
You must define an audience, purpose, and medium to tell your story.
You must have at least 6 visuals included that use the data provided.
Include a 250-500-word paper summarizing your audience, purpose, medium and design choices along with the above ethical consideration questions.


In [8]:
import pandas as pd
from pathlib import Path

# --- File Path ---
file_path = Path("MotherboardVICENewsKiaHyundaiTheftData.xlsx")
if not file_path.exists():
    raise FileNotFoundError(f"{file_path} not found in the working directory")

# --- Read Excel with Multi-Level Header ---
df = pd.read_excel(file_path, header=[0,1])

# --- Flatten columns into single strings ---
df.columns = [f"{a}_{b}".strip() if str(a) != 'nan' else b for a,b in df.columns]

# --- Melt into long format ---
df_long = df.melt(id_vars=[df.columns[0]],  # first column is Date
                  var_name='Location_Metric',
                  value_name='Value')

# --- Split the combined column into Location and Metric ---
df_long[['Location','Metric']] = df_long['Location_Metric'].str.split('_', expand=True)
df_long = df_long.drop(columns='Location_Metric')

# --- Pivot so each row is Date + Location ---
df_long = df_long.pivot_table(index=[df.columns[0],'Location'],
                              columns='Metric',
                              values='Value',
                              aggfunc='first').reset_index()

# --- Rename columns ---
df_long.columns.name = None
df_long = df_long.rename(columns={df.columns[0]: 'Date', 'Kia/Hyundais': 'Kia_Hundais'})

# --- Convert Percent to float ---
# Convert Percent to float safely
def percent_to_float(x):
    if pd.isna(x):
        return None
    if isinstance(x, str) and '%' in x:
        return float(x.rstrip('%')) / 100
    return float(x)

df_long['Percent'] = df_long['Percent'].apply(percent_to_float)

# --- Save CSV ---
df_long.to_csv('tidy_data.csv', index=False)
print("Saved tidy_data.csv")


Saved tidy_data.csv
